## UAV Intrusion Detection Stacked Ensemble
This notebook uses One-Class SVM, Autoencoder, and Local Outlier Factor as base models, and combines their outputs with a meta-classifier ensemble.

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPRegressor
import time
from sklearn.metrics import classification_report, accuracy_score, f1_score
import os

In [5]:
# Constants
DATA_PATH = "../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx"
MODEL_NAME = "StackedEnsemble_RF"
DATASET_NAME = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
RESULTS_CSV = "../../../2D_Results.csv"

### Load the Dataset

In [ ]:
# Load the dataset
try:
    df = pd.read_excel("../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx")
except FileNotFoundError:
    print("Error: The file '../Datasets/excel/GPS_Data_Simplified_2D_Feature_Map.xlsx' was not found.")
    print("Please make sure the path is correct.")
    exit()
df.head()

### Preprocessing

In [4]:
# Prepare features and labels
X = df.drop('Output', axis=1)
y = df['Output']

# Use benign data to train unsupervised models
X_train_raw = X[y == 0]
X_test_raw = X
y_test = y

# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# PCA for OC-SVM and LOF (85% variance)
pca_svm_lof = PCA(n_components=0.85, random_state=1082)
X_train_pca_svm_lof = pca_svm_lof.fit_transform(X_train_scaled)
X_test_pca_svm_lof = pca_svm_lof.transform(X_test_scaled)

# PCA for Autoencoder (95% variance)
pca_ae = PCA(n_components=0.95, random_state=1082)
X_train_pca_ae = pca_ae.fit_transform(X_train_scaled)
X_test_pca_ae = pca_ae.transform(X_test_scaled)

### Train Base Models

In [5]:
start_time_total = time.time()

# One-Class SVM
ocsvm = OneClassSVM(nu=0.011, gamma=0.000211)
ocsvm.fit(X_train_pca_svm_lof)
ocsvm_score = ocsvm.decision_function(X_test_pca_svm_lof)  # anomaly score

# Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=1000, novelty=True)
lof.fit(X_train_pca_svm_lof)
lof_score = lof.decision_function(X_test_pca_svm_lof)  # anomaly score

# Autoencoder using MLP
ae = MLPRegressor(hidden_layer_sizes=(3,), activation='relu', max_iter=100, random_state=1082)
ae.fit(X_train_pca_ae, X_train_pca_ae)
recon = ae.predict(X_test_pca_ae)
ae_score = np.mean((X_test_pca_ae - recon) ** 2, axis=1)  # reconstruction error

MemoryError: Unable to allocate 2.96 GiB for an array with shape (397825000,) and data type int64

### Meta-Classifier Ensemble

In [ ]:
# Stack raw anomaly scores
stacked_features = np.vstack([ocsvm_score, lof_score, ae_score]).T

# Meta-classifier (Random Forest) for multiclass classification
meta_clf = RandomForestClassifier(random_state=1082)
meta_clf.fit(stacked_features, y_test)
stacked_pred = meta_clf.predict(stacked_features)

total_train_time = time.time() - start_time_total

In [11]:
# Evaluation
report = classification_report(
    y_test,
    stacked_pred,
    labels=[0, 1, 2, 3],
    target_names=["benign", "attack1", "attack2", "attack3"],
    zero_division=0,
    output_dict=True
)
f1 = report["weighted avg"]["f1-score"]
accuracy = accuracy_score(y_test, stacked_pred)
print(classification_report(
    y_test,
    stacked_pred,
    labels=[0, 1, 2, 3],
    target_names=["benign", "attack1", "attack2", "attack3"],
    zero_division=0
))
print(f"F1 Score: {f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")

              precision    recall  f1-score   support

      benign       0.99      1.00      0.99    397825
     attack1       0.98      0.95      0.97     36458
     attack2       1.00      0.99      1.00     44232
     attack3       0.99      0.95      0.97     32015

    accuracy                           0.99    510530
   macro avg       0.99      0.97      0.98    510530
weighted avg       0.99      0.99      0.99    510530

F1 Score: 0.9907
Accuracy: 0.9908


In [14]:
# Save results to CSV
results = [{
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "f1score": f1,
    "accuracy": accuracy,
    "train_time_sec": total_train_time,
    "parameters": str(meta_clf.get_params()),
    "num_epochs": "N/A"
}]
results_df = pd.DataFrame(results)

file_exists = os.path.isfile(RESULTS_CSV)
results_df.to_csv(
    RESULTS_CSV,
    mode='a',
    header=not file_exists,
    index=False
)
print(f"Results appended to {RESULTS_CSV}")

Results appended to ../../../2D_Results.csv
